# Classification Problem

## Library Import

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')

## Data Import

The data was taken from the ["Don't Get Kicked!"](https://www.kaggle.com/competitions/DontGetKicked/) competition

In [ ]:
df = pd.read_csv('data/training.csv')
df.head()

In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
df_preprocessed = df.copy()

# Drop index column
df_preprocessed = df_preprocessed.drop('RefId', axis=1)

In [ ]:
# Transform `PurchDate` to datetime and sort by it
df_preprocessed['PurchDate'] = pd.to_datetime(df_preprocessed['PurchDate'], format='%m/%d/%Y')
df_preprocessed = df_preprocessed.sort_values(by='PurchDate')

## EDA

### NaNs research

In [ ]:
# Look at the NaNs percentage in the columns
missing_percent = df_preprocessed.isna().mean() * 100
missing_with_data = missing_percent[missing_percent > 0]
missing_with_data.sort_values(ascending=False)

### MMR features research

In [ ]:
# There is a lot of MMR cols. Look at the them in a more detailed way
MMR_cols = list(filter(lambda x: x.startswith('MMR'), df_preprocessed.columns))
df_preprocessed[MMR_cols].corr(numeric_only=True)

All of them are quite correlated

#### Values Counts

In [ ]:
df_preprocessed['WheelType'].value_counts()

In [ ]:
df_preprocessed['WheelTypeID'].value_counts()

In [ ]:
df_preprocessed['Trim'].value_counts()

In [ ]:
df_preprocessed['Size'].value_counts()

In [ ]:
df_preprocessed['TopThreeAmericanName'].value_counts()

In [ ]:
df_preprocessed['Nationality'].value_counts()

In [ ]:
df_preprocessed['Transmission'].value_counts()

## Data Preprocessing

### Features dropping

In [ ]:
# As was researched before all MMR features are quite correlated.
# Will use only one of them: `MMRAcquisitionRetailAveragePrice`.
# This will also solve the problem with NaNs in all columns starting with MMR string

MMR_to_drop = ['MMRAcquisitionAuctionAveragePrice', 'MMRAcquisitionAuctionCleanPrice', 'MMRAcquisitonRetailCleanPrice',
               'MMRCurrentAuctionAveragePrice', 'MMRCurrentAuctionCleanPrice', 'MMRCurrentRetailAveragePrice', 'MMRCurrentRetailCleanPrice']
df_preprocessed = df_preprocessed.drop(MMR_to_drop, axis=1)

In [ ]:
# Drop features `AUCGUART` and `PRIMEUNIT`. Most of theirs values are NaNs
df_preprocessed = df_preprocessed.drop(['PRIMEUNIT', 'AUCGUART'], axis=1)

### Values replacement

In [ ]:
# Also change `Manual` into `MANUAL` value in `Transmission` column. Prhaps error in data and won't be leakage
df_preprocessed['Transmission'] = df_preprocessed['Transmission'].replace({'Manual': 'MANUAL'})

In [ ]:
# 'Trim' feature has many single values. Change their category on 'other'
min_count = 100
small_categories = df_preprocessed['Trim'].value_counts()[df_preprocessed['Trim'].value_counts() < min_count].index
df_preprocessed['Trim'] = df_preprocessed['Trim'].replace(small_categories, 'Other')

df_preprocessed['Trim'].value_counts()

### Train/validate/test splitting

In [ ]:
# Make splitting by `PurchDate` column.
# Use the first 1/3 of dates for the train, the last 1/3 of dates for the test, and the middle 1/3 for the validation set

date_33_perc, date_66_perc = df_preprocessed['PurchDate'].quantile(0.33), df_preprocessed['PurchDate'].quantile(0.66)

df_train, df_validate, df_test = (df_preprocessed[df_preprocessed['PurchDate'] <= date_33_perc].reset_index(drop=True),
                                  df_preprocessed[(df_preprocessed['PurchDate'] > date_33_perc) & (df_preprocessed['PurchDate'] <= date_66_perc)].reset_index(drop=True),
                                  df_preprocessed[df_preprocessed['PurchDate'] > date_66_perc].reset_index(drop=True))

In [ ]:
print(f'Train data length: {len(df_train)}')
print(f'Validation data length: {len(df_validate)}')
print(f'Test data length: {len(df_test)}')
print('==========================')
print(f'Train data min `PurchDate`: {df_train['PurchDate'].min()}; Train data max `PurchDate`: {df_train['PurchDate'].max()}')
print(f'Validation data min `PurchDate`: {df_validate['PurchDate'].min()}; Validation data max `PurchDate`: {df_validate['PurchDate'].max()}')
print(f'Test data min `PurchDate`: {df_test['PurchDate'].min()}; Test data max `PurchDate`: {df_test['PurchDate'].max()}')

### NaNs fullfilling

In [ ]:
def ultimate_nan_fullfill(df):
    '''
    Fill with the most common value from train samples
    '''
    df_to_transform = df.copy()
    
    df_to_transform['WheelType'] = df_to_transform['WheelType'].fillna(df_train['WheelType'].mode()[0])
    df_to_transform['WheelTypeID'] = df_to_transform['WheelTypeID'].fillna(df_train['WheelTypeID'].mode()[0])
    df_to_transform['Trim'] = df_to_transform['Trim'].fillna(df_train['Trim'].mode()[0])
    df_to_transform['Size'] = df_to_transform['Size'].fillna(df_train['Size'].mode()[0])
    df_to_transform['TopThreeAmericanName'] = df_to_transform['TopThreeAmericanName'].fillna(df_train['TopThreeAmericanName'].mode()[0])
    df_to_transform['Nationality'] = df_to_transform['Nationality'].fillna(df_train['Nationality'].mode()[0])
    df_to_transform['Transmission'] = df_to_transform['Transmission'].fillna(df_train['Transmission'].mode()[0])
    
    df_to_transform['Color'] = df_to_transform['Color'].fillna(df_train['Color'].mode()[0])
    df_to_transform['SubModel'] = df_to_transform['SubModel'].fillna(df_train['SubModel'].mode()[0])
    df_to_transform['MMRAcquisitionRetailAveragePrice'] = df_to_transform['MMRAcquisitionRetailAveragePrice'].fillna(df_train['MMRAcquisitionRetailAveragePrice'].mode()[0])
    
    return df_to_transform

In [ ]:
df_train = ultimate_nan_fullfill(df_train)
df_validate = ultimate_nan_fullfill(df_validate)
df_test = ultimate_nan_fullfill(df_test)

In [ ]:
# Calculate NaNs count by columns
print(f'Train: {df_train.isna().sum().sum()}')
print(f'Validation: {df_validate.isna().sum().sum()}')
print(f'Test: {df_test.isna().sum().sum()}')

### One-Hot Encoding and Count Encoding

Will use One-Hot Encoding for columns: `Auction`, `Transmission`, `WheelType`, `Nationality`, `TopThreeAmericanName`

Will use Count Encoding for columns: `Make`, `Model`, `Trim`, `SubModel`, `Color`, `Size`, `VNST` because of their large amount of possible and rare values

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from category_encoders.count import CountEncoder

In [ ]:
# Cols for encoding
cols_to_count_encode = ['Make', 'Model', 'Trim', 'SubModel', 'Color', 'Size', 'VNST']
cols_to_onehot_encode = ['Auction', 'Transmission', 'WheelType', 'Nationality', 'TopThreeAmericanName']

In [ ]:
count_encoder = CountEncoder(cols=cols_to_count_encode, handle_unknown=-1)

onehot_encoder = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=False,
    dtype=int
)

Count Encoder application

In [ ]:
# Apply Count Encoder
df_train_encoded = count_encoder.fit_transform(df_train)
df_validate_encoded = count_encoder.transform(df_validate)
df_test_encoded = count_encoder.transform(df_test)

One-Hot-Encoder Encoder application

In [ ]:
def auto_onehot_encoder(cols_to_onehot_encode,
                        df_to_train, df_to_transform
                        ):
    """
    Function for automatic One-Hot Encoding
    """
    # Fit on train data
    onehot_encoder.fit(df_to_train[cols_to_onehot_encode])

    # Get all possible categories
    all_categories = []
    for i, col in enumerate(cols_to_onehot_encode):
        categories = onehot_encoder.categories_[i]
        all_categories.extend([f"{col}_{cat}" for cat in categories])

    # Function to transform with all cols keeping
    encoded = onehot_encoder.transform(df_to_transform[cols_to_onehot_encode])
    df_encoded = pd.DataFrame(encoded, columns=onehot_encoder.get_feature_names_out(cols_to_onehot_encode))

    # Add missing cols (for new categories)
    for col in all_categories:
        if col not in df_encoded.columns:
            df_encoded[col] = 0

    # Concat new cols and drop old
    df_transformed = pd.concat([df_to_transform, df_encoded], axis=1)
    df_transformed = df_transformed.drop(cols_to_onehot_encode, axis=1)
    
    return df_transformed

In [ ]:
df_train_encoded = auto_onehot_encoder(cols_to_onehot_encode, df_train, df_train_encoded)
df_validate_encoded = auto_onehot_encoder(cols_to_onehot_encode, df_train, df_validate_encoded)
df_test_encoded = auto_onehot_encoder(cols_to_onehot_encode, df_train, df_test_encoded)

In [ ]:
# Compare the new the dfs' shapes
print(f'df_train shape: {df_train.shape}')
print(f'df_validate shape: {df_validate.shape}')
print(f'df_test shape: {df_test.shape}')
print('=================================')
print(f'df_train_encoded shape: {df_train_encoded.shape}')
print(f'df_validate_encoded shape: {df_validate_encoded.shape}')
print(f'df_test_encoded shape: {df_test_encoded.shape}')

### X/y splitting

In [ ]:
X_train_encoded = df_train_encoded.drop(['IsBadBuy', 'PurchDate'], axis=1)
X_validate_encoded = df_validate_encoded.drop(['IsBadBuy', 'PurchDate'], axis=1)
X_test_encoded = df_test_encoded.drop(['IsBadBuy', 'PurchDate'], axis=1)

In [ ]:
X_train_with_cats = df_train.drop(['IsBadBuy', 'PurchDate'], axis=1)
X_validate_with_cats = df_validate.drop(['IsBadBuy', 'PurchDate'], axis=1)
X_test_with_cats = df_test.drop(['IsBadBuy', 'PurchDate'], axis=1)

In [ ]:
y_train = df_train['IsBadBuy']
y_validate = df_validate['IsBadBuy']
y_test = df_test['IsBadBuy']

### Normalization

Apply normalization to the new features

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
# With encoded categorical features

scaler = StandardScaler()
X_train_encoded_scaled = pd.DataFrame(scaler.fit_transform(X_train_encoded), columns=X_train_encoded.columns)
X_validate_encoded_scaled = pd.DataFrame(scaler.transform(X_validate_encoded), columns=X_validate_encoded.columns)
X_test_encoded_scaled = pd.DataFrame(scaler.transform(X_test_encoded), columns=X_test_encoded.columns)

In [ ]:
# With non-encoded categorical features

noncategorical_cols = X_train_with_cats.select_dtypes(['int64', 'float64']).columns
categorical_cols = X_train_with_cats.select_dtypes('category').columns

nonencoded_scaler = StandardScaler()
nonencoded_scaler.fit(X_train_with_cats[noncategorical_cols])

def df_with_cats_scaler(df):
    """
    Implement StandardScaler only to numeric cols ignoring categorical ones. Returns transformed pd.DataFrame
    """
    cat_cols = df[categorical_cols]
    num_cols = pd.DataFrame(nonencoded_scaler.transform(df[noncategorical_cols]), columns=noncategorical_cols)
    
    return pd.concat([cat_cols, num_cols], axis=1)

X_train_with_cats_scaled = df_with_cats_scaler(X_train_with_cats)
X_validate_with_cats_scaled = df_with_cats_scaler(X_validate_with_cats)
X_test_with_cats_scaled = df_with_cats_scaler(X_test_with_cats)

## Scikit-learn models

Train sklearn models (`Logistic Regression`, `KNN` and `Naive Bayes`), plot the ROC curve and find the AUC scores for future comparing with custom implementations and the scores after feature engineering

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, ExtraTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone

from modules.classification_metrics import custom_roc_auc_score, custom_gini_coefficient

In [ ]:
# DataFrame for future metrics results
df_sklearn_custom_metrics = pd.DataFrame(columns=['Model Name', 'AUC score', 'Gini Coef'])

In [ ]:
def fit_predict_pipeline(X_train, y_train, X_validate, y_validate, model, model_name, metrics_df):
    """
    Function for fit/predict pipeline with printing metrics and ROC curve
    """
    cur_model = clone(model)
    cur_model.fit(X_train, y_train)
    proba_preds = cur_model.predict_proba(X_validate)[:, 1]
    
    auc_score = custom_roc_auc_score(y_validate, proba_preds, plot=True)
    gini_score = custom_gini_coefficient(y_validate, proba_preds)
    
    print(f'AUC score: {auc_score}')
    print(f'Gini coef: {gini_score}')
    metrics_df.loc[len(metrics_df)] = [model_name, auc_score, gini_score]

### Logistic Regression

In [ ]:
fit_predict_pipeline(X_train_encoded_scaled, y_train, X_validate_encoded_scaled, y_validate,
                     LogisticRegression(), 'Logistic Regression (One-Hot + Count Encoders)',
                     df_sklearn_custom_metrics)

### KNN

In [ ]:
fit_predict_pipeline(X_train_encoded_scaled, y_train, X_validate_encoded_scaled, y_validate,
                     KNeighborsClassifier(), 'KNN (One-Hot + Count Encoders)',
                     df_sklearn_custom_metrics)

### GaussianNB (Naive Bayes)

In [ ]:
fit_predict_pipeline(X_train_encoded_scaled, y_train, X_validate_encoded_scaled, y_validate,
                     GaussianNB(), 'Naive Bayes (One-Hot + Count Encoders)',
                     df_sklearn_custom_metrics)

### Decision Tree

In [ ]:
fit_predict_pipeline(X_train_encoded_scaled, y_train, X_validate_encoded_scaled, y_validate,
                     DecisionTreeClassifier(), 'Decision Tree (One-Hot + Count Encoders)',
                     df_sklearn_custom_metrics)

### Extra Randomized Tree

In [ ]:
fit_predict_pipeline(X_train_encoded_scaled, y_train, X_validate_encoded_scaled, y_validate,
                     ExtraTreeClassifier(), 'Extra Randomized Tree (One-Hot + Count Encoders)',
                     df_sklearn_custom_metrics)

### Random Forest Classifier

In [ ]:
fit_predict_pipeline(X_train_encoded_scaled, y_train, X_validate_encoded_scaled, y_validate,
                     RandomForestClassifier(), 'Random Forest (One-Hot + Count Encoders)',
                     df_sklearn_custom_metrics)

## Boosting Ensembles

In [ ]:
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

### XGBoost (with pre-encoded categorical features)

In [ ]:
fit_predict_pipeline(X_train_encoded_scaled, y_train, X_validate_encoded_scaled, y_validate,
                     XGBClassifier(), 'XGBoost (One-Hot + Count Encoders)',
                     df_sklearn_custom_metrics)

### LightGBM (with pre-encoded categorical features)

In [ ]:
fit_predict_pipeline(X_train_encoded_scaled, y_train, X_validate_encoded_scaled, y_validate,
                     LGBMClassifier(), 'LightGBM (One-Hot + Count Encoders)',
                     df_sklearn_custom_metrics)

### CatBoost (with pre-encoded categorical features)

In [ ]:
fit_predict_pipeline(X_train_encoded_scaled, y_train, X_validate_encoded_scaled, y_validate,
                     CatBoostClassifier(verbose=False), 'CatBoost (One-Hot + Count Encoders)',
                     df_sklearn_custom_metrics)

### CatBoost (without pre-encoded categorical features)

In [ ]:
cat_cols = cols_to_count_encode + cols_to_onehot_encode

cat_boost_model = CatBoostClassifier(cat_features=cat_cols, verbose=False)

# Use the manual `fit_predict_pipeline` functionality because CatBoost loses its `cat_features` with sklearn.clone method
cat_boost_model.fit(X_train_with_cats_scaled, y_train)
proba_preds = cat_boost_model.predict_proba(X_validate_with_cats_scaled)[:, 1]
auc_score = custom_roc_auc_score(y_validate, proba_preds, plot=True)
gini_score = custom_gini_coefficient(y_validate, proba_preds)
print(f'AUC score: {auc_score}')
print(f'Gini coef: {gini_score}')
df_sklearn_custom_metrics.loc[len(df_sklearn_custom_metrics)] = ['CatBoost (df with non-encoded categorical)', auc_score, gini_score]

## Custom models' implementations

Now compare metrics values with custom impementations of `Logistic Regression`, `KNN` and `Naive Bayes`

In [ ]:
from modules.logistic_regression import Custom_LogisticRegression
from modules.knn import Custom_KNeighborsClassifier
from modules.naive_bayes import Custom_NaiveBayes
from modules.tree import Custom_DecisionTreeClassifier
from modules.extra_randomized_tree import Custom_ExtraTreesClassifier
from modules.random_forest import Custom_RandomForestClassifier
from modules.gradient_boosting import Custom_GradientBoostingClassifier

### Logistic Regression

In [ ]:
fit_predict_pipeline(X_train_encoded_scaled, y_train, X_validate_encoded_scaled, y_validate,
                     Custom_LogisticRegression(), 'Logistic Regression Custom (One-Hot + Count Encoders)',
                     df_sklearn_custom_metrics)

### KNN

In [ ]:
fit_predict_pipeline(X_train_encoded_scaled, y_train, X_validate_encoded_scaled, y_validate,
                     Custom_KNeighborsClassifier(), 'KNN Custom (One-Hot + Count Encoders)',
                     df_sklearn_custom_metrics)

### NaiveBayes

In [ ]:
fit_predict_pipeline(X_train_encoded_scaled, y_train, X_validate_encoded_scaled, y_validate,
                     Custom_NaiveBayes(), 'Naive Bayes Custom (One-Hot + Count Encoders)',
                     df_sklearn_custom_metrics)

### Decision Tree

In [ ]:
fit_predict_pipeline(X_train_encoded_scaled, y_train, X_validate_encoded_scaled, y_validate,
                     Custom_DecisionTreeClassifier(), 'Decision Tree Custom (One-Hot + Count Encoders)',
                     df_sklearn_custom_metrics)

### Extra Randomized Tree

In [ ]:
fit_predict_pipeline(X_train_encoded_scaled, y_train, X_validate_encoded_scaled, y_validate,
                     Custom_ExtraTreesClassifier(), 'Extra Randomized Tree Custom (One-Hot + Count Encoders)',
                     df_sklearn_custom_metrics)

### Random Forest

In [ ]:
fit_predict_pipeline(X_train_encoded_scaled, y_train, X_validate_encoded_scaled, y_validate,
                     Custom_RandomForestClassifier(), 'Random Forest Custom (One-Hot + Count Encoders)',
                     df_sklearn_custom_metrics)

### Gradient Boosting

In [ ]:
fit_predict_pipeline(X_train_encoded_scaled, y_train, X_validate_encoded_scaled, y_validate,
                     Custom_GradientBoostingClassifier(), 'Gradient Boosting Custom (One-Hot + Count Encoders)',
                     df_sklearn_custom_metrics)

## Scikit-learn and custom models comparison

In [ ]:
df_sklearn_custom_metrics

## GridSearch. Selecting the best model (algorithm + feature set)

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
final_model = CatBoostClassifier(verbose=False, random_state=42,
                                 border_count=128, boost_from_average=True,
                                 depth=6)
grid = {
    'iterations': [300, 500],
    'learning_rate': [0.01, 0.05, 0.1, 0.3],
    'grow_policy': ['SymmetricTree', 'Lossguide', 'Depthwise']
}

gridsearch_model = GridSearchCV(param_grid=grid,
                                estimator=final_model,
                                cv=5)

gridsearch_model.fit(X_train_with_cats_scaled, y_train, cat_features=cat_cols)

In [ ]:
pd.set_option('display.max_colwidth', None)
pd.DataFrame(gridsearch_model.cv_results_)[['rank_test_score', 'mean_test_score', 'params']].sort_values(by='rank_test_score').head()

In [ ]:
# Calculate the final predictions
final_train_preds = gridsearch_model.predict(X_train_with_cats_scaled)
final_valid_preds = gridsearch_model.predict(X_validate_with_cats_scaled)
final_test_preds = gridsearch_model.predict(X_test_with_cats_scaled)

final_train_proba_preds = gridsearch_model.predict_proba(X_train_with_cats_scaled)[:, 1]
final_valid_proba_preds = gridsearch_model.predict_proba(X_validate_with_cats_scaled)[:, 1]
final_test_proba_preds = gridsearch_model.predict_proba(X_test_with_cats_scaled)[:, 1]

## Calculate final predictions

Calculate the `Recall`, `Precision`, `F1-score` and `PR-AUC` and their custom variations on the train, validate and test data. Compare the results between sklearn and custom implementations and find out the best tracking metric for current problem.

In [ ]:
from sklearn.metrics import recall_score, precision_score, f1_score, average_precision_score
from modules.classification_metrics import custom_recall_score, custom_precision_score, custom_f1_score, custom_pr_auc_score

In [ ]:
df_final_metrics = pd.DataFrame(columns=['Samples Group', 'Custom Recall', 'Recall', 'Custom Precision', 'Precision',
                                         'Custom F1-Score', 'F1-Score', 'Custom PR-AUC', 'PR-AUC', 'ROC-AUC', 'Gini Coef'])
samples_dict = {'train': (y_train, final_train_preds, final_train_proba_preds),
                'validation': (y_validate, final_valid_preds, final_valid_proba_preds),
                'test': (y_test, final_test_preds, final_test_proba_preds)
                }

for group_name, samples in samples_dict.items():
    custom_recall = custom_recall_score(samples[0], samples[1])
    recall = recall_score(samples[0], samples[1])
    custom_precision = custom_precision_score(samples[0], samples[1])
    precision = precision_score(samples[0], samples[1])
    custom_f1score = custom_f1_score(samples[0], samples[1])
    f1score = f1_score(samples[0], samples[1])
    custom_pr_auc = custom_pr_auc_score(samples[0], samples[2])
    pr_auc = average_precision_score(samples[0], samples[2])
    roc_auc = custom_roc_auc_score(samples[0], samples[2])
    gini = custom_gini_coefficient(samples[0], samples[2])
    df_final_metrics.loc[len(df_final_metrics)] = [group_name, custom_recall, recall, custom_precision, precision,
                                                   custom_f1score, f1score, custom_pr_auc, pr_auc, roc_auc, gini]

### Final metrics comparison

In [ ]:
df_final_metrics

In [ ]:
final_test_preds

In [ ]:
gridsearch_model.predict_proba(X_test_with_cats_scaled)

### ROC-AUC graphs

In [ ]:
fig, axes = plt.subplots(ncols=3, dpi=200, figsize=(14, 6))

custom_roc_auc_score(y_train, final_train_proba_preds, plot=True, ax=axes[0])
custom_roc_auc_score(y_validate, final_valid_proba_preds, plot=True, ax=axes[1])
custom_roc_auc_score(y_test, final_test_proba_preds, plot=True, ax=axes[2])

plt.tight_layout()

### PR-AUC graphs

In [ ]:
fig, axes = plt.subplots(ncols=3, dpi=200, figsize=(14, 6))

custom_pr_auc_score(y_train, final_train_proba_preds, plot=True, ax=axes[0])
custom_pr_auc_score(y_validate, final_valid_proba_preds, plot=True, ax=axes[1])
custom_pr_auc_score(y_test, final_test_proba_preds, plot=True, ax=axes[2])

plt.tight_layout()